#Initialization

In [0]:
import pyspark.sql.functions as F

#Read Bronze Table

In [0]:
hospitals = spark.table("health.bronze.hospitals")

#Data Understanding

###Review Schema

In [0]:
hospitals.printSchema()
display(hospitals.limit(10))

#Data Profiling

###Check Null Values

In [0]:
display(
    hospitals.select(
        [
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in hospitals.columns
        ]
    )
)

###Whitespace Validation

In [0]:
string_cols = [
    field.name
    for field in hospitals.schema.fields
    if field.dataType.simpleString() == "string"
]

display(
    hospitals.select(
        [
            F.sum(
                F.when(
                    F.col(c) != F.trim(F.col(c)),
                    1
                ).otherwise(0)
            ).alias(c)
            for c in string_cols
        ]
    )
)

###Business Key Validation

In [0]:
total_rows = hospitals.count()

distinct_hospital_ids = (
    hospitals
    .select("hospital_id")
    .distinct()
    .count()
)

print(f"Total Rows: {total_rows}")
print(f"Distinct Hospital IDs: {distinct_hospital_ids}")

###State Validation

In [0]:
display(
    hospitals.groupBy("state")
    .count()
    .orderBy("state")
)

###Duplicate Validation


In [0]:
print(f"Total Rows: {hospitals.count()}")

print(
    f"Rows After dropDuplicates(): {hospitals.dropDuplicates().count()}"
)

#Transformations

###Create Clean Hospital Dataset

In [0]:
hospitals_clean = hospitals.dropDuplicates()

###Validation Summary

In [0]:
print(f"Clean Hospitals: {hospitals_clean.count()}")
print(f"Original Hospitals: {hospitals.count()}")

#Write Silver Tables

###Save Clean Hospitals

In [0]:
(
    hospitals_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.hospitals")
)

###Verify Silver Hospitals

In [0]:
display(
    spark.table("health.silver.hospitals")
    .limit(10)
)